In [3]:
from utils import *

n_sim = 1000
n = int(10/0.7)
B_RF = int(n * 0.7 )
seed = 42


data_generation_parameter_0_1 =   { 'shape_weibull': 1,  'p_1': -0.405, 'p_2': -0.4, 'p_3': -0.05, 'p_4': -0.01, 'n': n,
                                'scale_weibull_base':   22_080       , 
                                'rate_censoring':       0.00321    , 
                                'tau': 37, 
                                'data_type':           'weibull'               ,}  

X_pred_point = pd.DataFrame({'X_1': [0], 'X_2': [1], 'X_3': [80], 'X_4': [40]})
params_rf =         {   'n_estimators':B_RF,                        
                        'max_depth':4,
                        'min_samples_split':5,
                        'max_features': 'log2',
                        'random_state':  seed,
                        'weighted_bootstrapping': True, }

# Load the data
data = create_weibull_data(params=data_generation_parameter_0_1, random_state=seed)
df_train = create_data_with_ipc_weights(params=data_generation_parameter_0_1, data=data)

# Fit the model
params_rf["random_state"] = seed
clf = DecisionTreeBaggingClassifier(params_rf)
clf.fit(
    X=df_train.drop(
        ["time", "event", "weights_ipcw", "survived"], axis=1
    ).values,
    y=df_train["survived"].values,
    sample_weights=df_train["weights_ipcw"].values,
)

# Prediction für X_erwartung
_ ,rf_pred = clf.predict_proba(X_pred_point.values)
